# 03. Recommendation Baseline & Candidate Ranking
**Project**: Next-Best-Offer and Product Recommendation with Supply Constraints

Demonstrates the actual baseline recommendation approach, applying deterministic policy constraints to filter infeasible candidates, and showing sample outputs.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / 'data' / 'synthetic'
txns     = pd.read_csv(DATA_DIR / 'transactions.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
op_cat   = pd.read_csv(DATA_DIR / 'operational_catalogue.csv')

# Popularity Baseline — training on all data
from backend.app.recommendation.popularity import PopularityRanker

ranker = PopularityRanker(top_k=10)
ranker.fit(txns, item_col='product_id', order_col='transaction_id')

sample_recs = ranker.recommend(customer_history=set(), top_n=10)
rec_df = pd.DataFrame(sample_recs, columns=['product_id', 'popularity_score'])
rec_df = rec_df.merge(products[['product_id','product_name','category','product_status']], on='product_id')
rec_df = rec_df.merge(op_cat[['product_id','available_stock','gross_margin_pct']], on='product_id')
print('Top 10 Popularity Baseline Candidates (Unconstrained):')
print(rec_df.to_string(index=False))


### Apply Operational Policy Constraints

In [ ]:
feasible = rec_df[
    (rec_df['available_stock'] > 0) &
    (rec_df['gross_margin_pct'] >= 0.10) &
    (rec_df['product_status'] == 'active')
]
excluded = rec_df[~rec_df['product_id'].isin(feasible['product_id'])]

print(f'Total candidates: {len(rec_df)}')
print(f'Feasible:         {len(feasible)}')
print(f'Excluded:         {len(excluded)}')
if not excluded.empty:
    print('\nExcluded Products:')
    print(excluded[['product_id','product_name','available_stock','gross_margin_pct','product_status']].to_string(index=False))


### Live Decision Module Output for CUST_0001

In [ ]:
from backend.app.recommendation.decision_module import DecisionModule

dm = DecisionModule()
result = dm.decide('CUST_0001', top_k=5, reward_context=False)
print(f'Status: {result["status"]}')
print(f'Candidate Count: {result["candidate_count"]}')
print(f'Feasible Count:  {result["feasible_count"]}')
print(f'Excluded Count:  {result["excluded_count"]}')
print('\nTop-5 Recommendations:')
for i, rec in enumerate(result['recommendations'], 1):
    print(f'  {i}. {rec["product_name"]} [{rec["product_id"]}] score={rec["recommendation_score"]:.4f}')
